## Training notebook

In [1]:
import gym
import highway_env
import datetime

from stable_baselines3 import PPO

%load_ext tensorboard
import datetime, os
from ipywidgets import interact, widgets
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UNIX', 'WINDOWS'])

interactive(children=(Dropdown(description='desired_os', options=('UNIX', 'WINDOWS'), value='UNIX'), Output())…

<function __main__.f(desired_os)>

In [3]:
# Create and wrap the environment
env = gym.make("dm-env-v0")

OUTPUT_PATH = '/Users/fornerispighetti/HighwayDRL/training_output/' if os_id.value == "UNIX" else 'C:/Users/aless/OneDrive/Desktop/Thesis_repo_new/HighwayDRL/training_output/'

In [4]:
# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=False)

In [5]:
# PPO parameters
model = PPO('MlpPolicy', env, tensorboard_log=f'{OUTPUT_PATH}logdir', verbose=1)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [6]:
num_timesteps = 2.5e5

# Train the agent for n steps
model.learn(total_timesteps=int(num_timesteps), callback=[checkpoint_callback, eval_callback])

Logging to /Users/fornerispighetti/HighwayDRL/training_output/logdir/PPO_2


/Users/fornerispighetti/HighwayDRL/.venv/lib/python3.8/site-packages/stable_baselines3/common/evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=15.55 +/- 7.19
Episode length: 21.00 +/- 9.10
---------------------------------
| eval/              |          |
|    mean_ep_length  | 21       |
|    mean_reward     | 15.5     |
| time/              |          |
|    total_timesteps | 1000     |
---------------------------------
New best mean reward!


/Users/fornerispighetti/HighwayDRL/.venv/lib/python3.8/site-packages/stable_baselines3/common/evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=2000, episode_reward=17.86 +/- 10.22
Episode length: 25.00 +/- 13.54
---------------------------------
| eval/              |          |
|    mean_ep_length  | 25       |
|    mean_reward     | 17.9     |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 19.3     |
|    ep_rew_mean     | 14.8     |
| time/              |          |
|    fps             | 15       |
|    iterations      | 1        |
|    time_elapsed    | 128      |
|    total_timesteps | 2048     |
---------------------------------
Eval num_timesteps=3000, episode_reward=26.45 +/- 38.10
Episode length: 32.60 +/- 43.93
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 32.6        |
|    mean_reward          | 26.4        |
| time/                   |             |
|    total_tim

In [7]:
#Save the trained agent
model.save(f'{OUTPUT_PATH}models/ppo_RL_{str(os_id.value)}_2e5_{datetime.datetime.today().day}-{datetime.datetime.today().month}-{datetime.datetime.today().year}')

### Continued learning

In [5]:
model_cl = PPO.load("./models/ppo_RL_MacOS_4e4_26_5_2022_cl", env=env, tensorboard_log="./logdir")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [6]:
model_cl.learn(total_timesteps=int(2e4), reset_num_timesteps=False, callback=[checkpoint_callback, eval_callback])

Logging to ./logdir/PPO_20


/Users/fornerispighetti/HighwayDRL/.venv/lib/python3.8/site-packages/stable_baselines3/common/evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=41960, episode_reward=67.59 +/- 1.99
Episode length: 120.00 +/- 0.00
---------------------------------
| eval/              |          |
|    mean_ep_length  | 120      |
|    mean_reward     | 67.6     |
| time/              |          |
|    total_timesteps | 41960    |
---------------------------------
New best mean reward!
Eval num_timesteps=42960, episode_reward=67.75 +/- 0.90
Episode length: 120.00 +/- 0.00
---------------------------------
| eval/              |          |
|    mean_ep_length  | 120      |
|    mean_reward     | 67.7     |
| time/              |          |
|    total_timesteps | 42960    |
---------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 120      |
|    ep_rew_mean     | 82.3     |
| time/              |          |
|    fps             | 2        |
|    iterations      | 1        |
|    time_elapsed    | 865      |
|    total_timesteps | 43008    

In [7]:
model_cl.save(f'./models/ppo_RL_MacOS_6e4_{datetime.datetime.today().day}_{datetime.datetime.today().month}_{datetime.datetime.today().year}_cl')